# Retrieval Evaluation (Simple Model Loop)

This notebook evaluates retrieval quality across embedding setups with one fixed protocol:
- Embed each synthetic question (query).
- Retrieve top-5 chunks.
- Compute Hit Rate@5 and MRR@5.


In [12]:
from pathlib import Path
import json
import os

import chromadb
from dotenv import load_dotenv
from openai import OpenAI


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
CHROMA_PATH = PROJECT_ROOT / "backend" / "data" / "chroma_db"
GROUND_TRUTH_PATH = PROJECT_ROOT / "backend" / "data" / "eval" / "ground_truth_chunk_questions.jsonl"
BASE_COLLECTION_NAME = "estate_documents"

MODEL_RUNS = [
    {"model": "text-embedding-3-small", "collection": BASE_COLLECTION_NAME},
    {"model": "text-embedding-3-large", "collection": "estate_documents_eval_text_embedding_3_large"},
]
TOP_K = 5
BATCH_SIZE = 64
REBUILD_COLLECTIONS = False

load_dotenv(PROJECT_ROOT / ".env", override=True)
if not (os.getenv("OPENAI_API_KEY") or "").strip():
    raise ValueError("OPENAI_API_KEY is missing. Add it to /workspace/.env or your environment.")

client = chromadb.PersistentClient(path=str(CHROMA_PATH))
openai_client = OpenAI()

print(f"Project root: {PROJECT_ROOT}")
print(f"Chroma path: {CHROMA_PATH}")
print(f"Ground truth file: {GROUND_TRUTH_PATH}")
print(f"Model runs: {MODEL_RUNS}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Project root: /workspace
Chroma path: /workspace/backend/data/chroma_db
Ground truth file: /workspace/backend/data/eval/ground_truth_chunk_questions.jsonl
Model runs: [{'model': 'text-embedding-3-small', 'collection': 'estate_documents'}, {'model': 'text-embedding-3-large', 'collection': 'estate_documents_eval_text_embedding_3_large'}]


In [13]:
def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def fetch_all_chunks(chroma_collection, batch_size: int = 200) -> list[dict]:
    total = chroma_collection.count()
    rows: list[dict] = []

    for offset in range(0, total, batch_size):
        batch = chroma_collection.get(
            include=["documents", "metadatas"],
            limit=batch_size,
            offset=offset,
        )
        ids = batch.get("ids", [])
        docs = batch.get("documents", [])
        metas = batch.get("metadatas", [])

        for idx, chunk_id in enumerate(ids):
            rows.append(
                {
                    "chunk_id": chunk_id,
                    "text": docs[idx] or "",
                    "metadata": metas[idx] or {},
                }
            )

    return rows


def embed_texts(texts: list[str], model: str, batch_size: int = BATCH_SIZE) -> list[list[float]]:
    vectors: list[list[float]] = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        response = openai_client.embeddings.create(model=model, input=batch)
        vectors.extend(item.embedding for item in response.data)

    return vectors


def ensure_model_collection(
    *,
    model: str,
    collection_name: str,
    source_collection_name: str = BASE_COLLECTION_NAME,
    rebuild: bool = False,
    batch_size: int = BATCH_SIZE,
):
    source = client.get_or_create_collection(name=source_collection_name)
    target = client.get_or_create_collection(name=collection_name)

    should_rebuild = (
        collection_name != source_collection_name
        and (rebuild or target.count() == 0 or target.count() != source.count())
    )
    if not should_rebuild:
        return target

    client.delete_collection(name=collection_name)
    target = client.get_or_create_collection(name=collection_name)

    chunks = fetch_all_chunks(source)
    total = len(chunks)
    for start in range(0, total, batch_size):
        batch = chunks[start : start + batch_size]
        target.upsert(
            ids=[row["chunk_id"] for row in batch],
            documents=[row["text"] for row in batch],
            metadatas=[row["metadata"] for row in batch],
            embeddings=embed_texts([row["text"] for row in batch], model=model, batch_size=batch_size),
        )

        if (start // batch_size + 1) % 5 == 0 or start + len(batch) == total:
            print(f"Built {start + len(batch)}/{total} vectors for '{collection_name}'")

    return target


def reciprocal_rank(retrieved_ids: list[str], expected_id: str) -> tuple[float, int | None]:
    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id == expected_id:
            return 1.0 / rank, rank
    return 0.0, None


def evaluate_model(model: str, collection_name: str, ground_truth_rows: list[dict]) -> tuple[dict, list[dict]]:
    collection = ensure_model_collection(model=model, collection_name=collection_name, rebuild=REBUILD_COLLECTIONS)
    questions = [row["question"] for row in ground_truth_rows]
    query_embeddings = embed_texts(questions, model=model, batch_size=BATCH_SIZE)

    query_result = collection.query(query_embeddings=query_embeddings, n_results=TOP_K)
    retrieved_by_query = query_result.get("ids", [])

    scored_rows: list[dict] = []
    hit_sum = 0
    rr_sum = 0.0

    for item, retrieved_ids in zip(ground_truth_rows, retrieved_by_query):
        rr, rank = reciprocal_rank(retrieved_ids, item["expected_chunk_id"])
        hit = int(rank is not None)
        hit_sum += hit
        rr_sum += rr

        scored_rows.append(
            {
                **item,
                "retrieved_ids": retrieved_ids,
                "hit": hit,
                "rank": rank,
                "reciprocal_rank": rr,
            }
        )

    n = len(scored_rows) or 1
    summary = {
        "model": model,
        "collection": collection_name,
        "num_questions": len(scored_rows),
        "top_k": TOP_K,
        "hit_rate": hit_sum / n,
        "mrr": rr_sum / n,
    }
    return summary, scored_rows


In [14]:
ground_truth_rows = load_jsonl(GROUND_TRUTH_PATH)
if not ground_truth_rows:
    raise ValueError("Ground truth file is empty. Run notebooks/generate_ground_truth.ipynb first.")

print(f"Loaded {len(ground_truth_rows)} synthetic questions")

results = []
details_by_run = {}

for run in MODEL_RUNS:
    model_name = run["model"]
    collection_name = run["collection"]
    run_key = f"{model_name} @ {collection_name}"

    print("=" * 80)
    print(f"Evaluating: {run_key}")

    summary, rows = evaluate_model(model_name, collection_name, ground_truth_rows)
    results.append(summary)
    details_by_run[run_key] = rows

    print(f"hit_rate={summary['hit_rate']:.4f} | mrr={summary['mrr']:.4f}")

try:
    import pandas as pd

    results_df = pd.DataFrame(results).sort_values(
        by=["hit_rate", "mrr"],
        ascending=False,
    ).reset_index(drop=True)
    display(results_df)
except Exception:
    print(json.dumps(results, indent=2))


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loaded 465 synthetic questions
Evaluating: text-embedding-3-small @ estate_documents


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


hit_rate=0.7161 | mrr=0.4690
Evaluating: text-embedding-3-large @ estate_documents_eval_text_embedding_3_large


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Built 93/93 vectors for 'estate_documents_eval_text_embedding_3_large'


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


hit_rate=0.6968 | mrr=0.4720


,model,collection,num_questions,top_k,hit_rate,mrr
0,text-embedding-3-small,estate_documents,465,5,0.716129,0.469032
1,text-embedding-3-large,estate_documents_eval_text_embedding_3_large,465,5,0.696774,0.471971
